# Notebook for batch processes

## OpenAI

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

### Check the Status of a Batch

In [ ]:
batch_id = "batch_69a75361d2fc81908b45f011f26bdbeb"
batch = openai_client.batches.retrieve(batch_id)
print(batch)

In [ ]:
# List the 10 most recent batches
openai_client.batches.list(limit=10)

### Retrieve Results

Once the batch is complete, you can download the output by making a request against the Files API via the `output_file_id` field from the Batch object and writing it to a file on your machine, in this case `batch_output.jsonl`

In [ ]:
import json

def get_batch_results(batch_id: str) -> dict:
    """
    Fetches batch status and outputs (if completed)
    """
    batch = openai_client.batches.retrieve(batch_id)

    if batch.status != "completed":
        return {
            "status": batch.status,
            "message": "Batch not completed yet."
        }

    output_file_id = batch.output_file_id
    output_file = openai_client.files.content(output_file_id)

    results = {}
    for line in output_file.iter_lines():
        record = json.loads(line)
        custom_id = record["custom_id"]
        output_text = record["response"]["body"]["output"][1]["content"][0]["text"]
        results[custom_id] = output_text

    return results

In [ ]:
# results = get_batch_results(batch_id)

# model_name = "gpt-5.1"

# # save to local file
# local_filename = f"outputs/responses/{model_name}/batch_question_responses.jsonl"
# with open(local_filename, "w") as f:
#     for custom_id, output_text in results.items():
#         record = {
#             "custom_id": custom_id,
#             "output_text": output_text
#         }
#         f.write(json.dumps(record) + "\n")

In [ ]:
results = get_batch_results(batch_id)

model_name = "gpt-5.1"

# save to local file
local_filename = f"outputs/baseline_responses/{model_name}/batch_question_responses.jsonl"
with open(local_filename, "w") as f:
    for custom_id, output_text in results.items():
        record = {
            "custom_id": custom_id,
            "output_text": output_text
        }
        f.write(json.dumps(record) + "\n")

### Cancel a Batch Job

If necessary, can cancel an ongoing batch. The batch's status will change to `cancelling` until in-flight requests are complete (up to 10 minutes), after which the status will change to `cancelled`.

In [ ]:
batch_id = "batch_abc123"
openai_client.batches.cancel(batch_id)

## Anthropic's Claude

In [ ]:
import anthropic
from dotenv import load_dotenv
import os

load_dotenv(override=True)
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

### Monitor Batch Job Status

In [ ]:
# Automatically fetches more pages as needed.
for message_batch in anthropic_client.messages.batches.list(limit=20):
    print(message_batch)

In [ ]:
import time

MESSAGE_BATCH_ID = "msgbatch_013f5SixFJMFrFXHuVkXVoLZ"

message_batch = None
while True:
    message_batch = anthropic_client.messages.batches.retrieve(
        MESSAGE_BATCH_ID
    )
    if message_batch.processing_status == "ended":
        break

    print(f"Batch {MESSAGE_BATCH_ID} is still processing...")
    time.sleep(60)
print(message_batch)

### Retrieve Results

In [ ]:
MESSAGE_BATCH_ID = "msgbatch_013f5SixFJMFrFXHuVkXVoLZ"

# Stream results file in memory-efficient chunks, processing one at a time
for result in anthropic_client.messages.batches.results(MESSAGE_BATCH_ID,):
    match result.result.type:
        case "succeeded":
            print(f"Success! {result.custom_id}")
        case "errored":
            if result.result.error.type == "invalid_request":
                # Request body must be fixed before re-sending request
                print(f"Validation error {result.custom_id}")
            else:
                # Request can be retried directly
                print(f"Server error {result.custom_id}")
        case "expired":
            print(f"Request expired {result.custom_id}")

In [ ]:
import json

def get_batch_results(batch_name: str) -> dict:
    """
    Fetches batch status and outputs (if completed)
    """
    MESSAGE_BATCH_ID = "msgbatch_013f5SixFJMFrFXHuVkXVoLZ"

    results = {}
    for result in anthropic_client.messages.batches.results(MESSAGE_BATCH_ID,):
        custom_id = result.custom_id
        output_text = result.result.message.content[0].text
        results[custom_id] = output_text
    return results

In [ ]:
batch_id = "msgbatch_013f5SixFJMFrFXHuVkXVoLZ"
results = get_batch_results(batch_id)

model_name = "claude_4.5_sonnet"

# save to local file
local_filename = f"outputs/baseline_responses/{model_name}/batch_question_responses.jsonl"
# local_filename = f"outputs/responses/{model_name}/batch_question_responses.jsonl"
with open(local_filename, "w") as f:
    for custom_id, output_text in results.items():
        record = {
            "custom_id": custom_id,
            "output_text": output_text
        }
        f.write(json.dumps(record) + "\n")

### Cancel a Batch Job

In [ ]:
MESSAGE_BATCH_ID = "your_message_batch_id_here"

message_batch = anthropic_client.messages.batches.cancel(
    MESSAGE_BATCH_ID,
)
print(message_batch)

## Google's Gemini

In [4]:
from google import genai
from google.cloud import storage
from dotenv import load_dotenv
import os
import json

load_dotenv(override=True)
gemini_api_key = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=gemini_api_key)

project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
location = os.getenv("GOOGLE_CLOUD_REGION")
vertex_client = genai.Client(vertexai=True, project=project_id, location=location)
storage_client = storage.Client(project=project_id)

E0000 00:00:1772680925.265240 2186652 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1772680925.266802 2186652 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1772680925.266814 2186652 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1772680925.266817 2186652 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1772680925.266820 2186652 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


### Monitor Batch Job Status

#### GEMINI API

In [ ]:
import time

BATCH_ID = "batches/w65gdu7qkpwlfwjg0vob4ptkvik7ki3d3n4y"

print(f"Polling status for job: {BATCH_ID}")

# Poll the job status until it's completed.
while True:
    batch_job = gemini_client.batches.get(name=BATCH_ID)
    if batch_job.state.name in ('JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED'):
        break
    print(f"Job not finished. Current state: {batch_job.state.name}. Waiting 30 seconds...")
    time.sleep(30)

print(f"Job finished with state: {batch_job.state.name}")
if batch_job.state.name == 'JOB_STATE_FAILED':
    print(f"Error: {batch_job.error}")

In [ ]:
BATCH_ID = "batches/nloxrik70h7p4chlbd1er4i77033ff1thaao"

print(f"Retrieving status for job: {BATCH_ID}")

batch_job = gemini_client.batches.get(name=BATCH_ID)
print(f"Current state: {batch_job.state.name}")
print(f"Job details: {batch_job}")

#### VERTEX AI

In [ ]:
# using vertex ai client
BATCH_ID = "projects/2409567398/locations/global/batchPredictionJobs/3188536441550405632"
gcs_batch_job = vertex_client.batches.get(name=BATCH_ID)
gcs_batch_job

In [ ]:
# list all the batch prediction jobs in the project
for job in vertex_client.batches.list():
    print(job.name, job.create_time, job.state)

### Retrieve Results

#### GEMINI API

Once a file-based job succeeds, the results are written to an output file in the [Files API](./File_API.ipynb).

In [ ]:
if batch_job.state.name == 'JOB_STATE_SUCCEEDED':
    # The output is in another file.
    result_file_name = batch_job.dest.file_name
    print(f"Results are in file: {result_file_name}")

    print("\nDownloading and parsing result file content...")
    file_content_bytes = gemini_client.files.download(file=result_file_name)
    file_content = file_content_bytes.decode('utf-8')

    # The result file is also a JSONL file. Parse and print each line.
    for line in file_content.splitlines():
      if line:
        parsed_response = json.loads(line)
        # Pretty-print the JSON for readability
        print(json.dumps(parsed_response, indent=2))
        print("-" * 20)
else:
    print(f"Job did not succeed. Final state: {batch_job.state.name}")

In [ ]:
def get_batch_results(batch_name: str) -> dict:
    """
    Fetches batch status and outputs (if completed)
    """
    batch_job = gemini_client.batches.get(name=batch_name)

    # The output is in another file.
    result_file_name = batch_job.dest.file_name
    print(f"Results are in file: {result_file_name}")

    print("\nDownloading and parsing result file content...")
    file_content_bytes = gemini_client.files.download(file=result_file_name)
    file_content = file_content_bytes.decode('utf-8')

    results = {}
    for line in file_content.splitlines():
        if line:
            record = json.loads(line)
            custom_id = record["key"]
            response = record["response"]["candidates"][0]
            output_text = response["content"]["parts"][0]["text"]
            results[custom_id] = output_text
    return results

In [ ]:
results = get_batch_results(BATCH_ID)

model_name = "api-llama3.3"

# save to local file
local_filename = f"outputs/analysis/{model_name}_batch_analysis_responses.json"
with open(local_filename, "w") as f:
    f.write(json.dumps(results, indent=4))

#### VERTEX AI

In [1]:
from utils import load_jsonl_file
import json

results = {}
# download using the console, add to the same directory as this file, and just load the jsonl
path = "prediction-model-2026-03-06T01_51_19.814547Z_predictions.jsonl"

predictions = load_jsonl_file(path)
for line in predictions:
    custom_id = line["key"]
    output_text = line["response"]["candidates"][0]["content"]["parts"][0]["text"]
    results[custom_id] = output_text

In [2]:
model_name = "api-llama4"

# save to local file
local_filename = f"outputs/baseline_evaluation/{model_name}_batch_eval_responses.json"
with open(local_filename, "w") as f:
    f.write(json.dumps(results, indent=4))

In [ ]:
# import fsspec
# import json

# def get_vertex_batch_results(batch_job) -> dict:
#     """
#     Fetches batch results for a Vertex AI batch job (output in GCS).
#     """
#     fs = fsspec.filesystem("gcs")
#     file_paths = fs.glob(f"{batch_job.dest.gcs_uri}/*/predictions.jsonl")

#     results = {}
#     for path in file_paths:
#         print(path)
#         with fs.open(path, "r") as f:
#             for line in f:
#                 if line.strip():
#                     record = json.loads(line)
#                     custom_id = record["key"]
#                     output_text = record["response"]["candidates"][0]["content"]["parts"][0]["text"]
#                     results[custom_id] = output_text
#     return results

In [ ]:
# BATCH_ID = "projects/2409567398/locations/global/batchPredictionJobs/750505746398969856"
# gcs_batch_job = vertex_client.batches.get(name=BATCH_ID)

In [ ]:
# results = get_vertex_batch_results(gcs_batch_job)

# model_name = "api-llama4"

# # save to local file
# local_filename = f"outputs/analysis/{model_name}_batch_eval_responses.json"
# with open(local_filename, "w") as f:
#     f.write(json.dumps(results, indent=4))

### Cancel a Batch Job

In [ ]:
try:
  job_to_cancel_name = "batches/1pl7s602fkuog243rzt8gm09k87voy1slh2s" # <-- REPLACE THIS
  print(f"Attempting to cancel job: {job_to_cancel_name}")
  gemini_client.batches.cancel(name=job_to_cancel_name)
  print("Job cancellation request sent.")
except Exception as e:
  print(f"Error cancelling job: {e}")

### List Batch Jobs

#### GEMINI API

In [ ]:
batch_jobs = gemini_client.batches.list()

for batch_job in batch_jobs:
    print(batch_job)


#### VERTEX AI

In [ ]:
for job in vertex_client.batches.list():
    print(job.name, job.create_time, job.state)

### List Uploaded Files (GEMINI API)

In [ ]:
files = []
for f in gemini_client.files.list():
    print(' ', f.name)
    files.append(f.name)


### Delete uploaded files (GEMINI API)

In [ ]:
# files_to_delete = ["file_123", "file_456"]  # <-- REPLACE WITH YOUR FILE NAMES
files_to_delete = files


for file_name in files_to_delete:
    try:
        print(f"Attempting to delete file: {file_name}")
        gemini_client.files.delete(name=file_name)
    except Exception as e:
        print(f"Error deleting file {file_name}: {e}")

## Consolidating Batch Results to Final Responses Results (Positive and Negative)

In [ ]:
MODEL_NAME = "claude_4.5_sonnet" # "claude_4.5_sonnet" # "gpt-5.1"
batch_file = f"outputs/responses/{MODEL_NAME}/batch_question_responses.jsonl"
responses_file = f"outputs/responses/{MODEL_NAME}/question_responses.json"

In [ ]:
import json

# load batch file to dict
batch_responses = {}
with open(batch_file, "r") as f:
    for line in f:
        record = json.loads(line)
        custom_id = record["custom_id"]
        output_text = record["output_text"]
        batch_responses[custom_id] = output_text

# load responses file to list of dict
with open(responses_file, "r") as f:
    responses = json.load(f)

In [ ]:
for review in responses:
    review_id = review["ReviewID"]
    questions_answers = review["ModelGeneratedAnswersWithQuestions"]
    for key, value in questions_answers.items():
        positive_id = f"req-{review_id}_{key}_positive"
        negative_id = f"req-{review_id}_{key}_negative"
        if positive_id in batch_responses:
            review["ModelGeneratedAnswersWithQuestions"][key]["positive_answer"] = batch_responses[positive_id]
        if negative_id in batch_responses:
            review["ModelGeneratedAnswersWithQuestions"][key]["negative_answer"] = batch_responses[negative_id]

In [ ]:
# save responses to file
with open(responses_file, "w") as f:
    json.dump(responses, f, indent=2)

## Consolidating Batch Results to Final Responses Results (2 Sample of Positive)

In [ ]:
MODEL_NAME = "gpt-5.1" # "claude_4.5_sonnet" # "gpt-5.1"
batch_file = f"outputs/baseline_responses/{MODEL_NAME}/batch_question_responses.jsonl"
responses_file = f"outputs/baseline_responses/{MODEL_NAME}/positive_question_responses.json"

In [ ]:
import json

# load batch file to dict
batch_responses = {}
with open(batch_file, "r") as f:
    for line in f:
        record = json.loads(line)
        custom_id = record["custom_id"]
        output_text = record["output_text"]
        batch_responses[custom_id] = output_text

# load responses file to list of dict
with open(responses_file, "r") as f:
    responses = json.load(f)

In [ ]:
for review in responses:
    review_id = review["ReviewID"]
    questions_answers = review["ModelGeneratedAnswersWithQuestions"]
    for key, value in questions_answers.items():
        positive1_id = f"req-{review_id}_{key}_positive1"
        positive2_id = f"req-{review_id}_{key}_positive2"
        if positive1_id in batch_responses:
            review["ModelGeneratedAnswersWithQuestions"][key]["positive1_answer"] = batch_responses[positive1_id]
        if positive2_id in batch_responses:
            review["ModelGeneratedAnswersWithQuestions"][key]["positive2_answer"] = batch_responses[positive2_id]

In [ ]:
# save responses to file
with open(responses_file, "w") as f:
    json.dump(responses, f, indent=2)

## Consolidating Batch Eval Results to Final Evaluation Results

In [10]:
models = [
    "gpt-5.1",
    "claude_4.5_sonnet",
    "api-llama3.3",
    "api-llama4",
    "huatuo-7B",
    # "huatuo-8B",
    "qwen3-4B",
    # "qwen3-30B"
]

In [11]:
import json
import re

def extract_answer(text: str) -> str:
    """
    Extracts only nswer if answer exists else return the full text

    :param text: string of text

    :return: str
    """
    lower_text = text.lower()
    # parts = lower_text.split(delimiter, 1)
    parts = re.split(r"\*\*answer\*\*:|\*answer\*:", lower_text, maxsplit=1)

    # If len is > 1, delimiter existed; return second part, stripped of whitespace
    if len(parts) > 1:
        return parts[1].strip()

    return lower_text

def create_final_eval_results(model_name):
    batch_file = f"outputs/evaluation/{model_name}_batch_eval_responses.json"
    analysis_file = f"outputs/evaluation/{model_name}_eval_results.json"

    # load batch file to dict
    batch_responses = {}
    with open(batch_file, "r") as f:
        batch_responses = json.load(f)

    # load responses file to list of dict
    with open(analysis_file, "r") as f:
        analyses = json.load(f)

    for key, analysis in analyses.items():
        for q_type in ["positive", "negative"]:
            # direction
            direction_id = f"req-{key}_{q_type}_direction"

            if direction_id in batch_responses:
                if q_type == "positive":
                    analysis["first_response_metrics"]["evidence_direction"] = extract_answer(batch_responses[direction_id])
                elif q_type == "negative":
                    analysis["second_response_metrics"]["evidence_direction"] = extract_answer(batch_responses[direction_id])
    # save responses to file
    with open(analysis_file, "w") as f:
        json.dump(analyses, f, indent=4)

def create_final_eval_results_baseline(model_name):
    batch_file = f"outputs/baseline_evaluation/{model_name}_batch_eval_responses.json"
    analysis_file = f"outputs/baseline_evaluation/{model_name}_eval_results.json"

    # load batch file to dict
    batch_responses = {}
    with open(batch_file, "r") as f:
        batch_responses = json.load(f)

    # load responses file to list of dict
    with open(analysis_file, "r") as f:
        analyses = json.load(f)

    for key, analysis in analyses.items():
        for q_type in ["positive1", "positive2"]:
            # direction
            direction_id = f"req-{key}_{q_type}_direction"

            if direction_id in batch_responses:
                if q_type == "positive1":
                    analysis["first_response_metrics"]["evidence_direction"] = extract_answer(batch_responses[direction_id])
                elif q_type == "positive2":
                    analysis["second_response_metrics"]["evidence_direction"] = extract_answer(batch_responses[direction_id])
    # save responses to file
    with open(analysis_file, "w") as f:
        json.dump(analyses, f, indent=4)

In [12]:
for model_name in models:
    create_final_eval_results(model_name)

In [13]:
for model_name in models:
    create_final_eval_results_baseline(model_name)

## Alphabetize batch response

In [3]:
from utils import load_json_file
import json

model_names = [
    "gpt-5.1", "claude_4.5_sonnet", "api-llama3.3", "api-llama4",
    "qwen3-4B", "huatuo-7B", 
]

# "huatuo-8B"
# "qwen3-30B"
# "huatuo-70B"

def alphabetize_batch_responses(model_names, data_dir='outputs/baseline_evaluation'):
    for model in model_names:
        file_path = f'{data_dir}/{model}_batch_eval_responses.json'
        try:
            data = load_json_file(file_path)
            # Alphabetize the dictionary by keys
            sorted_data = dict(sorted(data.items()))
            # Save back the sorted dictionary
            with open(file_path, 'w') as file:
                json.dump(sorted_data, file, indent=4)
        except Exception as e:
            print(f"Error loading {file_path}: {e}")

In [4]:
alphabetize_batch_responses(model_names)

# EXTRA - combine two response files

In [ ]:
first_file = "outputs/responses/qwen3_thinking-4B/question_responses.json"
second_file = "outputs/responses/qwen3_thinking-4B/question_responses_1.json"


import json

# load batch file to dict
batch_responses = {}
with open(first_file, "r") as f:
    first = json.load(f)

# load responses file to list of dict
with open(second_file, "r") as f:
    second = json.load(f)


combined = first + second

print(len(combined))

In [ ]:
# Convert the list of dictionaries to a DataFrame
import pandas as pd
df = pd.DataFrame(data)

df.drop_duplicates(subset=['ReviewID'], keep='last', inplace=True)
all = df.to_dict("records")

In [ ]:
print(len(all))

In [ ]:
with open("outputs/responses/qwen3_thinking-4B/question_responses_combined.json", "w") as f:
    json.dump(all, f, indent=4)

In [ ]:
with open("outputs/responses/qwen3_thinking-4B/question_responses_combined.json", "r") as f:
    data = json.load(f)

In [ ]:
len(data)

In [ ]:
# Convert the list of dictionaries to a DataFrame
import pandas as pd
df = pd.DataFrame(data)

# Find rows that are duplicates based on the 'ReviewID' column
# keep=False marks all occurrences of a duplicate as True
duplicate_rows = df[df.duplicated(subset=['ReviewID'], keep=False)]

print("Duplicate rows based on 'ReviewID' column:")
print(duplicate_rows)